# 00 — Buffer Concepts

A buffer converts a **position** into an **area**. Given a point and a distance, a buffer answers the question: *what is the region within that distance of this point?*

```
point  +  radius  →  circular zone
line   +  radius  →  corridor zone
```

Buffers are used everywhere: blast radii, evacuation zones, flight corridors, signal coverage areas. This module builds them from scratch using the distance tools from module 06.

## Setup

In [ ]:
import math
from ipyleaflet import Map, GeoJSON, basemaps

## What is a Buffer?

A buffer is the set of all points within a given distance of a feature. Around a point, that set forms a **disk**. Around a line, it forms a **corridor**.

```
  Point buffer             Line buffer

    . . . .               . . . . . . . .
   .       .             .               .
  .    ●    .            .  A ────── B   .
   .       .             .               .
    . . . .               . . . . . . . .

  radius = r             width = r on each side
```

In spatial data, a buffer is always stored as a **polygon** — a closed ring of vertices that approximates the true circular or corridor shape. The more vertices, the smoother it looks.

## Circle Around a Point

To approximate a circle, we sample points at equal angular intervals around the center, each at exactly `radius` distance in that direction.

```
for angle in [0°, 15°, 30°, ... 345°]:
    point = destination(center, bearing=angle, distance=radius)

connect all points → polygon ring
```

With 24 points you get a recognizable circle. With 64 points it is smooth enough for any practical purpose.

In [ ]:
# Visualize the concept: a 24-point polygon approximating a circle
# Using a naive degree-offset approach just to show the shape
import math

center_lon, center_lat = 46.675, 24.688   # Riyadh
degree_radius = 3.0                        # ~300 km in rough degrees
n_points = 24

ring = []
for i in range(n_points):
    angle = 2 * math.pi * i / n_points
    ring.append([
        center_lon + degree_radius * math.cos(angle),
        center_lat + degree_radius * math.sin(angle)
    ])
ring.append(ring[0])   # close

circle_fc = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"label": "24-point polygon"},
            "geometry": {"type": "Polygon", "coordinates": [ring]}
        },
        {
            "type": "Feature",
            "properties": {"label": "center"},
            "geometry": {"type": "Point", "coordinates": [center_lon, center_lat]}
        }
    ]
}

m = Map(center=(center_lat, center_lon), zoom=5, basemap=basemaps.CartoDB.Positron)
m.add(GeoJSON(data=circle_fc,
              style={"color": "#e74c3c", "fillColor": "#e74c3c", "fillOpacity": 0.2, "weight": 2}))
m

## Corridor Around a Line

Buffering a line segment applies the same circular idea to every point along the segment. Each point on the line is the center of a small circle of the buffer radius — the union of all those circles forms the corridor.

In practice we approximate this by:
1. Sampling many points along the line at short intervals
2. Drawing a small circular buffer around each sample point
3. Overlapping them closely enough that they merge visually

In [ ]:
# Conceptual corridor: sample 20 points along a segment, draw circles at each
p1 = [-77.009, 38.889]   # Washington D.C.
p2 = [ 10.000, 50.000]   # Central Europe

n_samples = 20
degree_r  = 2.0
n_pts     = 16   # low count just for concept sketch

corridor_features = []
for i in range(n_samples + 1):
    t = i / n_samples
    cx = p1[0] + t * (p2[0] - p1[0])
    cy = p1[1] + t * (p2[1] - p1[1])
    ring = []
    for j in range(n_pts):
        angle = 2 * math.pi * j / n_pts
        ring.append([cx + degree_r * math.cos(angle), cy + degree_r * math.sin(angle)])
    ring.append(ring[0])
    corridor_features.append({
        "type": "Feature",
        "properties": {},
        "geometry": {"type": "Polygon", "coordinates": [ring]}
    })

# Add the path itself
corridor_features.append({
    "type": "Feature", "properties": {},
    "geometry": {"type": "LineString", "coordinates": [p1, p2]}
})

m2 = Map(center=(44, -30), zoom=3, basemap=basemaps.CartoDB.Positron)
m2.add(GeoJSON(
    data={"type": "FeatureCollection", "features": corridor_features},
    style={"color": "#2980b9", "fillColor": "#2980b9", "fillOpacity": 0.15, "weight": 1}
))
m2

## Units Matter

The examples above used **degree offsets** — simple arithmetic, but inaccurate. One degree of longitude is not the same distance everywhere on Earth.

| Latitude | 1° longitude ≈ |
|---|---|
| 0° (equator) | 111 km |
| 30° | 96 km |
| 45° | 78 km |
| 60° | 56 km |
| 90° (pole) | 0 km |

One degree of **latitude** is always ≈ 111 km. This asymmetry means a degree-based buffer circle is actually an oval — stretched in the latitude direction relative to longitude at high latitudes.

Notebooks 01 and 05 address this by using the **haversine + destination point** method to compute buffers in real kilometers.

## Real-World Analogies

| Context | Feature | Buffer radius | What the buffer models |
|---|---|---|---|
| Military | Target point | 5–50 km | Blast / lethal radius |
| Emergency mgmt | Incident point | 1–10 km | Evacuation zone |
| Telecom | Cell tower | 2–20 km | Signal coverage area |
| Aviation | Flight path | 5–50 km | Airspace corridor |
| Environmental | Spill point | 0.5–5 km | Contamination zone |

The radius is a **decision** — it encodes domain knowledge about the phenomenon being modeled. A 5 km evacuation radius is not computed from physics; it is a policy choice. Buffers make that choice visible and spatial.

## Exercise A

Using the concept circle code from the teaching cell, modify it to draw **two concentric circles** around Riyadh (`[46.675, 24.688]`):

- Inner ring: `degree_radius = 1.5`, 32 points, red
- Outer ring: `degree_radius = 3.0`, 32 points, orange

Add both as separate `GeoJSON` layers on the same map.

In [ ]:
from ipyleaflet import Map, GeoJSON, basemaps

# Draw two concentric degree-offset circles around Riyadh
# Inner (1.5°, 32 pts, red) and outer (3.0°, 32 pts, orange)
# Your code here

## Exercise B

Using the corridor sketch code, draw a corridor along the path from **Tehran** (`[51.388, 35.695]`) to **Moscow** (`[37.617, 55.755]`) with a `degree_radius` of `1.5` and `30` sample points.

Display the corridor and the path line on the same map. Use a different color for the path vs the corridor.

In [ ]:
from ipyleaflet import Map, GeoJSON, basemaps

# Draw a corridor along Tehran → Moscow
# degree_radius=1.5, 30 sample points, 16-pt circles
# Display corridor + path line with different colors
# Your code here

---

## Check Your Understanding

**1.** What determines the size of a buffer, and what determines its shape?

**2.** Why is a buffer always stored as an area (polygon) rather than as a circle (a mathematical object)?

```python
# No code needed — answer in your own words
```

## Next

In [01 — Buffering Points](./01_Buffering_Points.ipynb), we replace the naive degree-offset approach with a proper kilometer-accurate buffer using the haversine destination-point formula.